# 02 · Cabify — ¿Tráfico legítimo o artificial? (clasificación binaria)

**Problema:** bots que piden códigos SMS de verificación en masa (AIT, *Artificial Inflation of Traffic*) cuestan dinero. Hay que decidir, solicitud a solicitud, si **permitir o bloquear**.

**Qué vas a ver paso a paso**

1. Generar 4.000 solicitudes sintéticas con bots evidentes, bots «lentos y discretos» y usuarios genuinos con mala cobertura.
2. Explorar cómo se separan las señales.
3. Entrenar una regresión logística **iteración a iteración** y ver moverse la frontera de decisión.
4. Elegir el **umbral** de bloqueo mirando la matriz de confusión y el coste de negocio.
5. Explicar una solicitud concreta con contribuciones exactas en log-odds.

> Línea base didáctica independiente con datos sintéticos (semilla 42). **No es Infobip Signals ni el sistema de Cabify.** Reproduce el cálculo de `cabify/es/tools/generate-cabify-demo.py`.

In [1]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown

import innoit_style as inno
from innoit_style import MORADO, CORAL, TEAL, LAVANDA, LAVANDA_OSCURA, GRIS, TEXTO_SUAVE, PALETA, cascada
inno.aplicar()   # colores, tipografía Urbane y mapas de color de InnoIT

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.model_selection import train_test_split

## 1 · Generar solicitudes sintéticas

Cuatro señales por solicitud:

| Señal | Genuino típico | Bot típico |
|---|---|---|
| Solicitudes por minuto desde el mismo origen | ~2 | ~14 |
| % de códigos enviados que se introducen | ~86 % | ~28 % |
| Ruta de alto coste (destino SMS caro) | 5 % | 55 % |
| Dispositivo desconocido | 24 % | 65 % |

Dos complicaciones crean solapamiento realista: el 35 % de los bots imitan a humanos y el 10 % de los genuinos reintentan por mala cobertura.

In [2]:
rng = np.random.default_rng(42)
n = 4000
bot = rng.random(n) < 0.18
stealth = bot & (rng.random(n) < .35)
noisy = ~bot & (rng.random(n) < .10)
burst = np.where(bot, rng.lognormal(np.log(14), .75, n), rng.lognormal(np.log(1.8), .6, n))
burst = np.where(stealth, rng.lognormal(np.log(3.2), .6, n), burst)
burst = np.where(noisy, rng.lognormal(np.log(5.5), .55, n), burst)
burst = np.clip(np.rint(burst), 1, 60)
completion = np.where(bot, rng.normal(28, 24, n), rng.normal(86, 11, n))
completion = np.where(stealth, rng.normal(62, 22, n), completion)
completion = np.where(noisy, rng.normal(52, 26, n), completion)
completion = np.clip(np.rint(completion), 0, 100)
high_cost = np.where(bot, rng.random(n) < .55, rng.random(n) < .05).astype(float)
high_cost = np.where(stealth, rng.random(n) < .25, high_cost)
new_device = np.where(bot, rng.random(n) < .65, rng.random(n) < .24).astype(float)

X = np.column_stack([np.log1p(burst), completion / 100, high_cost, new_device])
y = bot.astype(int)
NOMBRES = ['ln(1+solic./min)', 'Códigos introducidos', 'Ruta alto coste', 'Dispositivo nuevo']
tipo = np.select([stealth, bot, noisy], ['Bot discreto', 'Bot', 'Genuino con mala cobertura'], 'Genuino')
df = pd.DataFrame(dict(solic_min=burst, pct_codigos=completion, alto_coste=high_cost.astype(bool),
                       disp_nuevo=new_device.astype(bool), tipo=tipo, es_bot=bot))
print(f'{bot.mean():.1%} de las solicitudes son artificiales')
display(df.head(8))
df.groupby('tipo')[['solic_min', 'pct_codigos', 'alto_coste', 'disp_nuevo']].mean().round(2)

18.6% de las solicitudes son artificiales


,solic_min,pct_codigos,alto_coste,disp_nuevo,tipo,es_bot
0,3.0,90.0,False,False,Genuino,False
1,2.0,86.0,False,True,Genuino,False
2,2.0,83.0,False,False,Genuino,False
3,9.0,11.0,False,False,Genuino con mala cobertura,False
4,7.0,21.0,False,False,Bot,True
5,1.0,87.0,False,False,Genuino,False
6,1.0,88.0,False,False,Genuino,False
7,1.0,95.0,False,False,Genuino,False


,solic_min,pct_codigos,alto_coste,disp_nuevo
tipo,,,,
Bot,19.07,30.32,0.54,0.63
Bot discreto,3.75,61.66,0.23,0.67
Genuino,2.24,85.55,0.05,0.23
Genuino con mala cobertura,6.35,48.47,0.08,0.24


## 2 · Explorar las señales

Elige una señal: ninguna separa perfectamente a bots y genuinos por sí sola. Por eso hace falta combinarlas en un modelo.

In [3]:
def explorar(señal='Solicitudes por minuto'):
    fig, ax = plt.subplots(figsize=(8, 3.6))
    if señal in ('Solicitudes por minuto', '% códigos introducidos'):
        v = burst if señal == 'Solicitudes por minuto' else completion
        bins = np.logspace(0, np.log10(60), 30) if señal == 'Solicitudes por minuto' else np.linspace(0, 100, 30)
        ax.hist(v[~bot], bins, alpha=.6, color=TEAL, label='Genuino', density=True)
        ax.hist(v[bot], bins, alpha=.6, color=CORAL, label='Bot', density=True)
        if señal == 'Solicitudes por minuto': ax.set_xscale('log')
        ax.set(xlabel=señal, ylabel='Densidad')
    else:
        v = high_cost if señal == 'Ruta de alto coste' else new_device
        tasas = [v[~bot].mean(), v[bot].mean()]
        ax.bar(['Genuino', 'Bot'], tasas, color=[TEAL, CORAL])
        for i, t in enumerate(tasas): ax.text(i, t, f'{t:.0%}', ha='center', va='bottom')
        ax.set(ylabel=f'% con «{señal}»')
    ax.set_title(señal); ax.legend() if ax.get_legend_handles_labels()[0] else None
    plt.show()

widgets.interact(explorar, señal=widgets.ToggleButtons(
    options=['Solicitudes por minuto', '% códigos introducidos', 'Ruta de alto coste', 'Dispositivo nuevo'], description='Señal'));

interactive(children=(ToggleButtons(description='Señal', options=('Solicitudes por minuto', '% códigos introdu…

## 3 · Entrenar la regresión logística paso a paso

El modelo calcula `z = b + w·x` y lo convierte en probabilidad con la sigmoide `p = 1 / (1 + e^{-z})`. El descenso de gradiente ajusta `w` y `b` para reducir la pérdida logarítmica.

En el gráfico de la derecha, la línea es la **frontera p = 0,5** sobre las dos señales continuas (las binarias se fijan en su media). Pulsa ▶.

In [4]:
train, test = train_test_split(np.arange(n), test_size=.25, random_state=42, stratify=y)
mu, sd = X[train].mean(0), X[train].std(0)
Xs = (X[train] - mu) / sd
sigm = lambda z: 1 / (1 + np.exp(-z))

w, b, lr, hist = np.zeros(4), 0., 2., []
GUARDAR = set(np.unique(np.geomspace(1, 3000, 70).astype(int))) | {0}
for it in range(3001):
    p = sigm(Xs @ w + b)
    if it in GUARDAR:
        loss = -np.mean(y[train]*np.log(p + 1e-12) + (1 - y[train])*np.log(1 - p + 1e-12))
        coef = w / sd; icpt = b - (w * mu / sd).sum()            # pesos en unidades originales
        auc = roc_auc_score(y[test], sigm(X[test] @ coef + icpt)) if it else .5
        hist.append((it, coef.copy(), icpt, loss, auc))
    g = p - y[train]
    w -= lr * (Xs.T @ g / len(train) + w / sd**2 / len(train))  # L2 (C = 1) sobre los pesos originales, como sklearn
    b -= lr * g.mean()

muestra = rng.choice(train, 700, replace=False)

def dibujar_entreno(paso):
    it, coef, icpt, loss, auc = hist[paso]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
    a1.plot([h[0] for h in hist], [h[3] for h in hist], color=GRIS)
    a1.scatter([it], [loss], color=CORAL, zorder=3, s=50)
    a1.set_xscale('symlog', linthresh=1)
    a1.set(xlabel='Iteración (escala log)', ylabel='Pérdida logarítmica', title=f'Iteración {it} · pérdida {loss:.3f} · AUC prueba {auc:.3f}')
    a2.scatter(X[muestra, 0], X[muestra, 1], c=np.where(y[muestra], CORAL, TEAL), s=8, alpha=.5)
    xs = np.linspace(X[:, 0].min(), X[:, 0].max(), 50)
    if abs(coef[1]) > 1e-9:
        resto = icpt + coef[2]*mu[2] + coef[3]*mu[3]
        a2.plot(xs, -(resto + coef[0]*xs) / coef[1], color=MORADO, lw=2.5, label='Frontera p = 0,5')
        a2.legend(loc='lower left')
    a2.set(xlim=(xs[0] - .1, xs[-1] + .1), ylim=(-.05, 1.05), xlabel=NOMBRES[0], ylabel=NOMBRES[1],
           title='Teal = genuino · coral = bot')
    plt.tight_layout(); plt.show()
    display(pd.DataFrame({'peso': coef}, index=NOMBRES).T.round(3))

play = widgets.Play(min=0, max=len(hist) - 1, interval=200)
slider = widgets.IntSlider(min=0, max=len(hist) - 1, description='Paso')
widgets.jslink((play, 'value'), (slider, 'value'))
display(widgets.HBox([play, slider]), widgets.interactive_output(dibujar_entreno, {'paso': slider}))

Output()

Comprobación: el modelo de `scikit-learn` (el mismo que usa el explorador) llega a los mismos pesos.

In [5]:
model = LogisticRegression(C=1.0, max_iter=1000).fit(X[train], y[train])
p_test = model.predict_proba(X[test])[:, 1]
display(pd.DataFrame({'Descenso de gradiente': [hist[-1][2], *hist[-1][1]],
                      'scikit-learn': [model.intercept_[0], *model.coef_[0]]},
                     index=['Intercepto', *NOMBRES]).round(3))
print(f'AUC en prueba: {roc_auc_score(y[test], p_test):.3f}')

,Descenso de gradiente,scikit-learn
Intercepto,-2.349,-2.349
ln(1+solic./min),1.646,1.646
Códigos introducidos,-4.454,-4.457
Ruta alto coste,1.959,1.961
Dispositivo nuevo,1.899,1.900


AUC en prueba: 0.955


## 4 · Elegir el umbral: precisión, recall y coste

El modelo da una **probabilidad**; bloquear o no es una **decisión de negocio**. Cada bot no detectado cuesta 1,50 € (SMS pagado) y cada usuario genuino bloqueado cuesta lo que elijas (cliente perdido). Mueve el umbral y busca el más barato.

In [6]:
umbrales = np.linspace(.01, .99, 99)
yt = y[test]

def matriz(u):
    pred = p_test >= u
    return (int((pred & (yt == 1)).sum()), int((pred & (yt == 0)).sum()),
            int((~pred & (yt == 1)).sum()), int((~pred & (yt == 0)).sum()))

def umbral(umbral=.5, coste_bloqueo=5.0):
    tp, fp, fn, tn = matriz(umbral)
    coste = lambda u: (matriz(u)[1]*coste_bloqueo + matriz(u)[2]*1.5) / len(test) * 1000
    costes = np.array([coste(u) for u in umbrales])
    fig, axs = plt.subplots(1, 3, figsize=(14, 3.8), gridspec_kw=dict(width_ratios=[1.3, .8, 1.2]))
    bins = np.linspace(0, 1, 41)
    axs[0].hist(p_test[yt == 0], bins, color=TEAL, alpha=.6, label='Genuino')
    axs[0].hist(p_test[yt == 1], bins, color=CORAL, alpha=.6, label='Bot')
    axs[0].axvline(umbral, color=MORADO, lw=2); axs[0].set_yscale('log')
    axs[0].set(xlabel='Probabilidad de bot', ylabel='Solicitudes (log)', title='Puntuaciones en prueba'); axs[0].legend()
    m = np.array([[tn, fp], [fn, tp]])
    axs[1].imshow(m, cmap='inno'); axs[1].grid(False)
    for (i, j), v in np.ndenumerate(m):
        axs[1].text(j, i, v, ha='center', va='center', fontsize=13, color='white' if v > m.max()/2 else MORADO, fontweight=600)
    axs[1].set_xticks([0, 1], ['Permitir', 'Bloquear']); axs[1].set_yticks([0, 1], ['Genuino', 'Bot'])
    axs[1].set_title('Matriz de confusión')
    axs[2].plot(umbrales, costes, color=MORADO)
    axs[2].scatter([umbral], [coste(umbral)], color=MORADO, s=60, zorder=3, label='Umbral elegido')
    axs[2].scatter([umbrales[costes.argmin()]], [costes.min()], marker='*', s=200, color=CORAL, zorder=3,
                   label=f'Más barato ({umbrales[costes.argmin()]:.2f})')
    axs[2].set(xlabel='Umbral', ylabel='€ por 1.000 solicitudes', title='Coste de negocio'); axs[2].legend()
    plt.tight_layout(); plt.show()
    display(Markdown(f'**Precisión** {tp/max(1, tp+fp):.1%} (de lo bloqueado, cuánto era bot) · '
                     f'**Recall** {tp/max(1, tp+fn):.1%} (de los bots, cuántos paramos) · '
                     f'**Coste** {coste(umbral):.2f} € / 1.000 solicitudes'))

widgets.interact(umbral,
    umbral=widgets.FloatSlider(value=.5, min=.01, max=.99, step=.01, description='Umbral'),
    coste_bloqueo=widgets.ToggleButtons(options=[('2 €', 2.0), ('5 €', 5.0), ('10 €', 10.0)], value=5.0,
                                        description='Bloquear genuino'));

interactive(children=(FloatSlider(value=0.5, description='Umbral', max=0.99, min=0.01, step=0.01), ToggleButto…

## 5 · Explicar una solicitud

Construye una solicitud. La cascada parte de la **solicitud media** de entrenamiento y suma `coeficiente × (variable − media)` en log-odds. La suma final, pasada por la sigmoide, es exactamente la probabilidad del modelo: no hay aproximación.

In [7]:
coef, icpt = model.coef_[0], model.intercept_[0]
medias = X[train].mean(0)
base = float(icpt + coef @ medias)

def explicar(solic_min=12, pct_codigos=40, alto_coste=True, disp_nuevo=False, umbral=.5):
    x = np.array([np.log1p(solic_min), pct_codigos / 100, float(alto_coste), float(disp_nuevo)])
    contrib = coef * (x - medias)
    z = base + contrib.sum(); p = 1 / (1 + np.exp(-z))
    assert np.isclose(p, model.predict_proba(x[None])[0, 1])
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.8), gridspec_kw=dict(width_ratios=[1.2, 1]))
    cascada(a1, base, NOMBRES, contrib, 'Solicitud media', 'Esta solicitud', '{:.2f}')
    a1.set(ylabel='Log-odds de bot', title='Contribución de cada señal')
    zs = np.linspace(-8, 8, 200)
    a2.plot(zs, 1 / (1 + np.exp(-zs)), color=GRIS)
    a2.axhline(umbral, ls='--', color=MORADO, label=f'Umbral {umbral:.2f}')
    a2.scatter([z], [p], s=120, color=CORAL if p >= umbral else TEAL, zorder=3)
    a2.set(xlabel='Log-odds (z)', ylabel='Probabilidad', title=f'p(bot) = {p:.1%}'); a2.legend()
    plt.tight_layout(); plt.show()
    display(Markdown('### ' + ('🚫 Bloquear' if p >= umbral else '✅ Permitir')))

widgets.interact(explicar,
    solic_min=widgets.IntSlider(12, 1, 60, description='Solic./min'),
    pct_codigos=widgets.IntSlider(40, 0, 100, description='% códigos'),
    alto_coste=widgets.Checkbox(True, description='Ruta alto coste'),
    disp_nuevo=widgets.Checkbox(False, description='Dispositivo nuevo'),
    umbral=widgets.FloatSlider(value=.5, min=.01, max=.99, step=.01, description='Umbral'));

interactive(children=(IntSlider(value=12, description='Solic./min', max=60, min=1), IntSlider(value=40, descri…

## Para llevarse

- Un clasificador devuelve **riesgo**, no una sentencia: el umbral lo fija el coste de equivocarse.
- Precisión y recall tiran en sentidos opuestos; el coste de negocio decide el equilibrio.
- Los bots discretos son los que más cuestan: ahí es donde un modelo con más señales aportaría valor.